# Pears 6

### Importing things 

In [8]:
import statsmodels.api as sm
import numpy as np
import warnings
import sys 
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

script_path = os.path.abspath(os.path.join('..', 'scripts'))
if script_path not in sys.path:
    sys.path.append(script_path)

from spread import SPREAD

warnings.filterwarnings("ignore")

### Reading in data

In [9]:
months = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512"
]

my_files = [
    [f"../data/processed/audusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/audusd_dukascopy_bid_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_bid_{m}.parquet" for m in months]
]

# Bump volume since we have 3x the data
builder = SPREAD(threshold=1000) 
df = builder.build(my_files)

print(df.head(5))

built 15455 rows
                                   Asset_A   Asset_B     Log_A     Log_B  \
timestamp                                                                  
2025-01-02 10:20:10.133000+00:00  0.621755  0.561855 -0.475209 -0.576511   
2025-01-02 10:25:55.560000+00:00  0.621930  0.561855 -0.474928 -0.576511   
2025-01-02 10:32:05.618000+00:00  0.621680  0.561780 -0.475330 -0.576645   
2025-01-02 10:34:14.918000+00:00  0.621880  0.561780 -0.475008 -0.576645   
2025-01-02 10:38:25.397000+00:00  0.621690  0.561840 -0.475314 -0.576538   

                                  Return_A  Return_B  
timestamp                                             
2025-01-02 10:20:10.133000+00:00  0.000265  0.000027  
2025-01-02 10:25:55.560000+00:00  0.000281  0.000000  
2025-01-02 10:32:05.618000+00:00 -0.000402 -0.000133  
2025-01-02 10:34:14.918000+00:00  0.000322  0.000000  
2025-01-02 10:38:25.397000+00:00 -0.000306  0.000107  


### Running it 

In [10]:
print("--- 1. ENGLE-GRANGER COINTEGRATION ---")
# We MUST use levels (Log_A and Log_B) to find the cointegration ratio
Y = df['Log_A']
X = sm.add_constant(df['Log_B'])
ols_model = sm.OLS(Y, X).fit()

beta = ols_model.params['Log_B']
alpha = ols_model.params['const']
print(f"Hedge Ratio (Beta): {beta:.4f}")

# Calculate the actual Spread levels (We need this for the Z-Score trading signal later)
spread = Y - (beta * df['Log_B'] + alpha)

print("\n--- 2. VOLATILITY DIFFERENCING & SCALING ---")
# Because you upgraded the SPREAD class, we can calculate the spread differences 
# directly using your new Return columns!
spread_diff = df['Return_A'] - (beta * df['Return_B'])

# Scale to Basis Points
spread_diff_scaled = spread_diff * 10000 

# Add Jitter (Anti-Zero-Variance trick)
np.random.seed(42)
jitter = np.random.normal(0, 1e-4, size=len(spread_diff_scaled))
spread_diff_ready = spread_diff_scaled + jitter

print(f"Data prepared: {len(spread_diff_ready)} rows ready for optimizer.")

print("\n--- 3. MARKOV REGIME FITTING ---")
ms_model = sm.tsa.MarkovRegression(
    spread_diff_ready, 
    k_regimes=2, 
    switching_variance=True, 
    trend='c'
).fit(disp=False) 

print("Success: Markov Model fitted perfectly!")

--- 1. ENGLE-GRANGER COINTEGRATION ---
Hedge Ratio (Beta): 0.6161

--- 2. VOLATILITY DIFFERENCING & SCALING ---
Data prepared: 15455 rows ready for optimizer.

--- 3. MARKOV REGIME FITTING ---
Success: Markov Model fitted perfectly!


In [11]:
print(ms_model.summary())

                        Markov Switching Model Results                        
Dep. Variable:                      y   No. Observations:                15455
Model:               MarkovRegression   Log Likelihood              -44693.740
Date:                Wed, 22 Apr 2026   AIC                          89399.480
Time:                        11:22:38   BIC                          89445.354
Sample:                             0   HQIC                         89414.678
                              - 15455                                         
Covariance Type:               approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0067      0.030     -0.220      0.826      -0.066       0.053
sigma2        12.9795      0.187     69.431      0.0